# ICARE telescope-resource corpus — decision dossier (B)

This notebook is the methodology record for the project decisions that
turn the evidence in `A_eda.ipynb` and `A2_grandma_evidence.ipynb` into an
explicit policy for the future observational-resource layer. Every figure
below is recomputed here, fresh, from `data/interim/telescopes/` and
`data/interim/telescopes/reference/` only — never copied from a stored A
or A2 cell — so every number is one that was visible at the moment the
decision was taken. It does not implement normalization, build a final
resource table, or write any file; it decides what a later implementation
stage is allowed to do.


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/interim/telescopes").is_dir())
CAPTURE_ID = "capture_20260808_071334"
ICARE_INTERIM_DIR = ROOT / "data/interim/telescopes" / CAPTURE_ID
ICARE_RAW_DIR = ROOT / "data/raw/telescopes/icare" / CAPTURE_ID
GRANDMA_RAW_DIR = ROOT / "data/raw/reference/grandma"
GRANDMA_INTERIM_PATH = ROOT / "data/interim/telescopes/reference/grandma_table.parquet"

ICARE_TABLE_NAMES = ["telescopes", "instruments", "allocations", "observations"]
EXPECTED_ROWS = {"telescopes": 89, "instruments": 95, "allocations": 38, "observations": 93}

ICARE = {name: pd.read_parquet(ICARE_INTERIM_DIR / f"{name}.parquet") for name in ICARE_TABLE_NAMES}
GRANDMA = pd.read_parquet(GRANDMA_INTERIM_PATH)
with (GRANDMA_RAW_DIR / "manifest.json").open() as handle:
    grandma_source = json.load(handle)

print("tables loaded:")
for name, frame in ICARE.items():
    print(f"  ICARE {name:14s} rows={len(frame):3d} columns={frame.shape[1]:3d}")
print(f"  GRANDMA table      rows={len(GRANDMA):3d} columns={GRANDMA.shape[1]:3d}")

controls = pd.DataFrame(
    [{"table": n, "expected_rows": EXPECTED_ROWS[n], "observed_rows": len(ICARE[n]),
      "status": "PASS" if len(ICARE[n]) == EXPECTED_ROWS[n] else "FAIL"} for n in ICARE_TABLE_NAMES]
    + [{"table": "grandma_table", "expected_rows": 39, "observed_rows": len(GRANDMA),
       "status": "PASS" if len(GRANDMA) == 39 else "FAIL"}]
)
print("\nCONTROLS")
print(controls.to_string(index=False))
if (controls["status"] != "PASS").any():
    raise ValueError("control check failed:\n" + controls.to_string(index=False))


def has_content(series):
    """True where a cell holds real content: '', '-', '[]', '{}', null and NaN are empty."""
    empty_tokens = {"", "-", "[]", "{}"}
    def alive(value):
        if value is None:
            return False
        if isinstance(value, str):
            return value.strip() not in empty_tokens
        if isinstance(value, float) and np.isnan(value):
            return False
        return True
    return series.map(alive)


def tokenize(name):
    """Comparison-only name tokens: split on separators, drop single characters."""
    return {t.lower() for t in re.split(r"[/\-\s]+", name) if len(t) >= 2}


def token_overlap_score(g_tokens, i_tokens):
    exact = g_tokens & i_tokens
    fuzzy = {gt for gt in g_tokens for it in i_tokens if gt != it and (gt in it or it in gt)}
    return len(exact | fuzzy)


def parse_size(value):
    value = str(value).strip()
    if value in ("", "-"):
        return None
    match = re.search(r"[\d.]+", value)
    return float(match.group()) if match else None


# Known, unambiguous facility acronym relevant to this table (explicit
# "known alias" evidence, used only alongside name-token/aperture evidence).
KNOWN_ALIASES = {"CFHT": "Canada-France-Hawaii Telescope"}
MATCH_APERTURE_TOLERANCE_M = 0.05


def aperture_agrees(icare_name, g_size, tel_by_name):
    icare_dia = tel_by_name.get(icare_name)
    if icare_dia is None or pd.isna(icare_dia) or g_size is None:
        return None
    delta = round(abs(float(icare_dia) - float(g_size)), 3)
    return bool(delta <= MATCH_APERTURE_TOLERANCE_M)


def classify_grandma_matches(grandma_df, icare_telescopes):
    """Diagnostic ICARE<->GRANDMA name matching, per GRANDMA row (not per
    unique name, since the source reuses 'TNT' at two different
    observatories). Returns one row per GRANDMA record; never persisted."""
    icare_names = set(icare_telescopes["name"])
    tel_by_name = icare_telescopes.set_index("name")["diameter"]
    rows = []
    for _, row in grandma_df.iterrows():
        g_name = row["telescope_name"]
        g_size = parse_size(row["size_m"])
        if g_name in icare_names:
            rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                        "candidates": [g_name], "classification": "EXACT"})
            continue
        g_tokens = tokenize(g_name)
        scores = {}
        for i_name in icare_names:
            s = token_overlap_score(g_tokens, tokenize(i_name))
            for acronym, expansion in KNOWN_ALIASES.items():
                if acronym.lower() in g_tokens and i_name == expansion:
                    s += 2
            if s > 0:
                scores[i_name] = s
        if not scores:
            rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                        "candidates": [], "classification": "UNMATCHED"})
            continue
        max_score = max(scores.values())
        top = sorted(n for n, s in scores.items() if s == max_score)
        if len(top) == 1:
            agrees = aperture_agrees(top[0], g_size, tel_by_name)
            cls = "AMBIGUOUS" if agrees is False else "HIGH-CONFIDENCE CANDIDATE"
            rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                        "candidates": top, "classification": cls})
        else:
            agreeing = [c for c in top if aperture_agrees(c, g_size, tel_by_name) is True]
            if len(agreeing) == 1:
                rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                            "candidates": [agreeing[0]], "classification": "HIGH-CONFIDENCE CANDIDATE"})
            else:
                rows.append({"row_index_in_source": row["row_index_in_source"], "grandma_name": g_name,
                            "candidates": top, "classification": "AMBIGUOUS"})
    return pd.DataFrame(rows)


tables loaded:
  ICARE telescopes     rows= 89 columns= 21
  ICARE instruments    rows= 95 columns= 42
  ICARE allocations    rows= 38 columns= 50
  ICARE observations   rows= 93 columns= 27
  GRANDMA table      rows= 39 columns= 15

CONTROLS
        table  expected_rows  observed_rows status
   telescopes             89             89   PASS
  instruments             95             95   PASS
  allocations             38             38   PASS
 observations             93             93   PASS
grandma_table             39             39   PASS


## Decision 1 — resource identity and relationships

**Observed.** Every one of the four ICARE tables' own `id` columns is a
complete, duplicate-free key; all three structured foreign-key relations
(`instrument -> telescope`, `allocation -> instrument`,
`observation -> instrument`) resolve with zero dangling references.
Exactly one relation disagrees with a nested copy of itself: telescope
id=137 nests allocation id=78 in its serialized `instruments[].allocations`,
but no row with `id=78` exists in the standalone allocations table.
Separately, the accepted GRANDMA reference reuses the raw label `'TNT'`
for two different telescopes at two different observatories with two
different apertures — a raw name is not even unique *within* one external
source, let alone across sources.
**Why it matters.** A later implementation needs one unambiguous way to
say "this telescope" / "this instrument" that survives both internal
serialization quirks and external-source name reuse.
**Decided.** `telescopes` and `instruments` remain distinct entities (a
telescope hosts one or more instruments; nothing collapses them).
ICARE's own numeric `id` per resource is the sole canonical identity;
`name` is never used as an identity key internally. Nested embedded
relations (e.g. `telescope.instruments`, `telescope.allocations`) are
supporting/cross-check evidence only — the standalone resource capture is
canonical, and where the two disagree (as for telescope 137) the
discrepancy is recorded, not silently resolved in either direction.
Names, aliases, and location text are diagnostic evidence for *external*
source matching only (see Decision 3); they play no role in ICARE's own
identity.
**Scope.** 89 telescopes / 95 instruments / 38 allocations / 93
observations, 0 duplicate ids in any of the 4 tables, 0 dangling ids
across the 3 checked relations, 1 nested-vs-standalone discrepancy (of 38
allocations), 1 duplicated raw name across 39 GRANDMA rows.


In [2]:
tel, inst, alloc, obs = ICARE["telescopes"], ICARE["instruments"], ICARE["allocations"], ICARE["observations"]

identity = pd.DataFrame([
    {"table": name, "rows": len(df), "distinct_ids": df["id"].nunique(),
     "duplicate_ids": int(df["id"].duplicated().sum())}
    for name, df in ICARE.items()
])
print("identity check, all 4 tables:")
print(identity.to_string(index=False))

relations = pd.DataFrame([
    {"relation": "instrument -> telescope",
     "dangling": int((~inst["telescope_id"].isin(tel["id"])).sum()), "of": len(inst)},
    {"relation": "allocation -> instrument",
     "dangling": int((~alloc["instrument_id"].isin(inst["id"])).sum()), "of": len(alloc)},
    {"relation": "observation -> instrument",
     "dangling": int((~obs["instrument_id"].isin(inst["id"])).sum()), "of": len(obs)},
])
print("\nreferential integrity:")
print(relations.to_string(index=False))

with (ICARE_RAW_DIR / "telescopes.json").open() as handle:
    raw_telescopes = json.load(handle)["data"]
with (ICARE_RAW_DIR / "allocations.json").open() as handle:
    standalone_allocation_ids = {a["id"] for a in json.load(handle)["data"]}
telescope_137 = next(t for t in raw_telescopes if t["id"] == 137)
nested_allocation_ids = [a["id"] for i in telescope_137["instruments"] for a in i.get("allocations", [])]
print(f"\ntelescope id=137 nested allocation id(s): {nested_allocation_ids}")
print(f"present in standalone allocations table: "
      f"{[a for a in nested_allocation_ids if a in standalone_allocation_ids]}")
print(f"absent from standalone allocations table: "
      f"{[a for a in nested_allocation_ids if a not in standalone_allocation_ids]}")

duplicate_grandma_names = GRANDMA["telescope_name"].value_counts()
duplicate_grandma_names = duplicate_grandma_names[duplicate_grandma_names > 1]
print(f"\nGRANDMA raw telescope_name values reused across rows: {len(duplicate_grandma_names)}")
print(GRANDMA.loc[GRANDMA["telescope_name"].isin(duplicate_grandma_names.index),
                  ["row_index_in_source", "telescope_name", "location", "size_m"]].to_string(index=False))


identity check, all 4 tables:
       table  rows  distinct_ids  duplicate_ids
  telescopes    89            89              0
 instruments    95            95              0
 allocations    38            38              0
observations    93            93              0

referential integrity:
                 relation  dangling  of
  instrument -> telescope         0  95
 allocation -> instrument         0  38
observation -> instrument         0  93

telescope id=137 nested allocation id(s): [78]
present in standalone allocations table: []
absent from standalone allocations table: [78]

GRANDMA raw telescope_name values reused across rows: 1
 row_index_in_source telescope_name      location size_m
                   1            TNT Thai Nat Obs.   2.40
                   2            TNT Xinglong Obs.   0.80


## Decision 2 — GRANDMA matching confidence and enrichment eligibility

**Observed.** Recomputing the diagnostic ICARE<->GRANDMA name matching
from the current accepted reference, per GRANDMA row (not per unique
name, because of the repeated `'TNT'` label), yields four confidence
tiers over the 39 GRANDMA rows.
**Why it matters.** Every later GRANDMA-sourced decision (FOV, Mlim,
`Use`/`Rob`) needs one shared, already-decided answer to "which matched
rows may enrich ICARE automatically", instead of re-deciding matching
confidence per field.
**Decided.** Automatic enrichment is permitted only from `EXACT` and
`HIGH-CONFIDENCE CANDIDATE` rows. `AMBIGUOUS` and `UNMATCHED` rows
contribute no automatic enrichment; their evidence is retained as
supporting-only, for human review in a later stage, never silently
merged. Matching is always performed per raw GRANDMA row
(`row_index_in_source`), never by collapsing on `telescope_name` first,
because the source itself reuses one name for two different telescopes.
Every automatic enrichment carries its source row's provenance
(`COMPLEMENTARY_REFERENCE`, GRANDMA row index) so it can be traced back
to the exact source line.
**Scope.** 39 GRANDMA rows total; recomputed classification counts below
determine exactly how many are enrichment-eligible.


In [3]:
GRANDMA_MATCHES = classify_grandma_matches(GRANDMA, tel)
counts = GRANDMA_MATCHES["classification"].value_counts()
print("GRANDMA diagnostic matching, recomputed fresh:")
print(counts.to_string())

eligible_mask = GRANDMA_MATCHES["classification"].isin(["EXACT", "HIGH-CONFIDENCE CANDIDATE"])
GRANDMA_ELIGIBLE = GRANDMA_MATCHES[eligible_mask].copy()
GRANDMA_ELIGIBLE["icare_name"] = GRANDMA_ELIGIBLE["candidates"].map(lambda c: c[0])
print(f"\nenrichment-eligible GRANDMA rows (EXACT + HIGH-CONFIDENCE CANDIDATE): "
      f"{len(GRANDMA_ELIGIBLE)} of {len(GRANDMA_MATCHES)}")
print(f"excluded (AMBIGUOUS + UNMATCHED, supporting-only): "
      f"{len(GRANDMA_MATCHES) - len(GRANDMA_ELIGIBLE)} of {len(GRANDMA_MATCHES)}")

tel_id_by_name = tel.set_index("name")["id"]
instrument_count_by_telescope = inst.groupby("telescope_id").size()
GRANDMA_ELIGIBLE["n_icare_instruments"] = GRANDMA_ELIGIBLE["icare_name"].map(
    lambda n: int(instrument_count_by_telescope.get(tel_id_by_name.get(n), 0)))
single_instrument = int((GRANDMA_ELIGIBLE["n_icare_instruments"] == 1).sum())
multi_instrument = int((GRANDMA_ELIGIBLE["n_icare_instruments"] > 1).sum())
print(f"\nof the eligible matches: {single_instrument} map to a single-instrument ICARE telescope "
      f"(safe to associate at instrument level); {multi_instrument} map to a telescope with more "
      f"than one ICARE instrument (must stay telescope-level only, per Decision 7 below):")
print(GRANDMA_ELIGIBLE.loc[GRANDMA_ELIGIBLE["n_icare_instruments"] > 1,
                          ["grandma_name", "icare_name", "n_icare_instruments"]].to_string(index=False))


GRANDMA diagnostic matching, recomputed fresh:
classification
HIGH-CONFIDENCE CANDIDATE    23
EXACT                        10
AMBIGUOUS                     3
UNMATCHED                     3

enrichment-eligible GRANDMA rows (EXACT + HIGH-CONFIDENCE CANDIDATE): 33 of 39
excluded (AMBIGUOUS + UNMATCHED, supporting-only): 6 of 39

of the eligible matches: 26 map to a single-instrument ICARE telescope (safe to associate at instrument level); 7 map to a telescope with more than one ICARE instrument (must stay telescope-level only, per Decision 7 below):
        grandma_name                     icare_name  n_icare_instruments
       Xinglong-2.16                 Xinglong-2.16m                    2
             GMG-2.4                       GMG-2.4m                    2
         CFHT/WIRCam Canada-France-Hawaii Telescope                    2
        CFHT/MegaCam Canada-France-Hawaii Telescope                    2
            ShAO/T2m                       ShAO-T2m                    2
      G

## Decision 3 — evidence-provenance labels and conflict policy

**Observed.** Even restricted to the 33 GRANDMA rows Decision 2 judged
enrichment-eligible, comparing ICARE against GRANDMA for the same
telescope disagrees in measurable cases: aperture (`diameter` vs
`size_m`) and robotic status (`robotic` vs `rob`) both carry at least one
genuine `DIFFERS` case, plus several telescopes where GRANDMA's `rob`
value is not even a plain yes/no. The evidence gathered across A and A2
also comes from four genuinely different places: ICARE's own API, the
accepted GRANDMA reference, ICARE's historical observation log, and
(later) deterministic astronomy calculations.
**Why it matters.** Without an explicit label, neither the later
deterministic filter nor the LLM can tell whether a value is a live ICARE
field, an external reference figure, an empirical measurement from a
handful of instruments, or nothing at all — and a silent overwrite when
two sources disagree would hide a real, measured conflict.
**Decided.** Adopt five evidence-origin labels for any field that is
enriched or derived beyond a raw ICARE passthrough: `DIRECT` (supplied
directly by the ICARE API), `COMPLEMENTARY_REFERENCE` (the accepted
GRANDMA table), `EMPIRICAL` (supported by historical ICARE observations),
`DERIVED` (later deterministic astronomy logic), `UNKNOWN` (no
sufficiently reliable evidence). A raw ICARE field with no enrichment is
implicitly `DIRECT` and does not need the label repeated. When two labeled
values for the same concept disagree, both raw values and their labels
are preserved, a conflict flag is set, and the canonical value is left
`UNKNOWN` unless a specific decision below states an explicit, evidenced
precedence rule for that field. No field is silently overwritten.
**Scope.** Measured below on the 33 enrichment-eligible telescopes from
Decision 2; re-examined per field in Decisions 7, 8 and 10.


In [4]:
conflict_rows = []
for _, m in GRANDMA_ELIGIBLE.iterrows():
    grandma_row = GRANDMA[GRANDMA["row_index_in_source"] == m["row_index_in_source"]].iloc[0]
    icare_row = tel[tel["name"] == m["icare_name"]].iloc[0]
    g_size = parse_size(grandma_row["size_m"])
    icare_dia = icare_row["diameter"]
    size_status = "MISSING ON ONE SIDE"
    if icare_dia is not None and g_size is not None:
        size_status = "AGREES" if round(abs(icare_dia - g_size), 3) <= 0.005 else "DIFFERS"
    g_rob = grandma_row["rob"]
    icare_rob = bool(icare_row["robotic"])
    rob_status = "NOT DIRECTLY COMPARABLE" if g_rob not in ("yes", "no") else (
        "AGREES" if (g_rob == "yes") == icare_rob else "DIFFERS")
    conflict_rows.append({"grandma_name": m["grandma_name"], "icare_name": m["icare_name"],
                          "icare_diameter": icare_dia, "grandma_size": g_size, "size_status": size_status,
                          "icare_robotic": icare_rob, "grandma_rob": g_rob, "rob_status": rob_status})

conflicts = pd.DataFrame(conflict_rows)
print(f"size_status counts:\n{conflicts['size_status'].value_counts().to_string()}")
print(f"\nrob_status counts:\n{conflicts['rob_status'].value_counts().to_string()}")

print("\nrows where at least one of the two fields DIFFERS or is NOT DIRECTLY COMPARABLE:")
notable = conflicts[(conflicts["size_status"] != "AGREES") | (conflicts["rob_status"] != "AGREES")]
print(notable.to_string(index=False))


size_status counts:
size_status
AGREES                 27
MISSING ON ONE SIDE     4
DIFFERS                 2

rob_status counts:
rob_status
AGREES                     26
NOT DIRECTLY COMPARABLE     4
DIFFERS                     3

rows where at least one of the two fields DIFFERS or is NOT DIRECTLY COMPARABLE:
        grandma_name                     icare_name  icare_diameter  grandma_size         size_status  icare_robotic grandma_rob              rob_status
             TRT-SBO   Thai Robotic Telescope - SBO            0.70           0.7              AGREES          False         yes                 DIFFERS
       Les Makes/T60                  Les-Makes/T60            0.60           0.6              AGREES          False      remote NOT DIRECTLY COMPARABLE
           Zeiss-600              Terskol/Zeiss-600            0.60           0.6              AGREES          False          np NOT DIRECTLY COMPARABLE
                IRIS                       OHP/IRIS            0.50        

## Decision 4 — band representation

**Observed.** `instruments.band` carries 5 distinct raw labels. Casefolding
collapses `'Optical'` and `'optical'` into one category, spanning the
large majority of instruments; `'ir'` and `'Xray'` each stand alone under
casefolding (no collision with any other label); one instrument carries
`'Optical, IR'`, a single free-text cell naming two bands at once.
**Why it matters.** Grouping or filtering instruments by band would treat
`'Optical'` and `'optical'` as two different categories unless compared
case-insensitively, undercounting the dominant band by roughly a third.
**Decided.** The raw `band` value is never modified. A second,
comparison-normalized value (casefolded) is derived alongside it purely
for grouping/filtering (`DERIVED`, from `DIRECT` ICARE text) and is never
presented as if it were a distinct raw label. The one combined
`'Optical, IR'` cell is not split into two rows or two values; it is left
as a single raw label whose casefolded form does not collide with
anything else, so no special case is required. No attempt is made to
decide, from the label alone, whether the observed case difference is
purely lexical or hides a semantic difference (e.g. a stricter definition
of "optical" at one site) — that judgement is out of scope here.
**Scope.** 95 of 95 instruments keep their raw `band` value unchanged;
90 of 95 (68 `'Optical'` + 22 `'optical'`) collapse into one
casefolded category; 2 `'ir'`, 2 `'Xray'`, 1 `'Optical, IR'` remain
distinct under casefolding.


In [5]:
band_counts = inst["band"].value_counts(dropna=False)
print("raw band distribution:")
print(band_counts.to_string())

casefolded = inst["band"].astype(str).str.casefold()
casefold_counts = casefolded.value_counts()
print(f"\ndistinct raw labels: {inst['band'].nunique()} -> distinct casefolded labels: "
      f"{casefolded.nunique()}")
print(casefold_counts.to_string())

collision_groups = inst.groupby(casefolded)["band"].agg(lambda s: sorted(set(s)))
collisions = {k: v for k, v in collision_groups.items() if len(v) > 1}
print(f"\ncasefolded groups combining more than one raw spelling: {collisions}")
affected = int(inst["band"].astype(str).str.casefold().isin(collisions.keys()).sum())
print(f"instruments affected by the case-only collision: {affected} of {len(inst)}")


raw band distribution:
band
Optical        68
optical        22
ir              2
Xray            2
Optical, IR     1

distinct raw labels: 5 -> distinct casefolded labels: 4
band
optical        90
ir              2
xray            2
optical, ir     1

casefolded groups combining more than one raw spelling: {'optical': ['Optical', 'optical']}
instruments affected by the case-only collision: 90 of 95


## Decision 5 — filter representation

**Observed.** ICARE's `instruments.filters` is an instrument-level JSON
list; 91 of 95 instruments carry at least one entry, spanning 56 distinct
raw labels, with no case/whitespace variants among them. GRANDMA's
`filter` column is telescope-level free text in an unrelated naming
convention for its Photometry-section rows (e.g. `'BVRCIC'`), and a
wavelength range for its Spectroscopy-section rows. The 93 historical
ICARE observations use only 2 distinct `filt` labels
(`'bessellr'`, `'ps1::open'`), and both already appear in some
instrument's own `filters` list.
**Why it matters.** Filtering telescopes/instruments by available filter
must query one well-defined representation; conflating ICARE's
per-instrument codes with GRANDMA's per-telescope free text risks
attributing a filter to the wrong instrument on a multi-instrument
telescope, and inventing a cross-vocabulary mapping (e.g. Johnson-Cousins
letters to ICARE's prefixed codes) without explicit evidence risks being
wrong in exactly the cases that matter.
**Decided.** ICARE's instrument-level `filters` list is the primary,
queryable filter representation, kept as raw labels (`DIRECT`). GRANDMA's
`filter` value is retained only as `COMPLEMENTARY_REFERENCE` text tied to
its own matched row and section (Photometry vs Spectroscopy; see Decision
2 for which rows are enrichment-eligible), never merged into or used to
invent an ICARE instrument's filter list. No cross-vocabulary mapping
between the two naming conventions is attempted in this stage; unresolved
labels remain distinct, and an instrument's filter identity stays
`UNKNOWN` where ICARE reports none.
**Scope.** 91 of 95 ICARE instruments carry >=1 filter (56 distinct raw
labels, 0 case/whitespace variants); 32 of 39 GRANDMA rows carry a
Photometry-shaped Filter value; 2 distinct historical `filt` labels
across 93 observations, both already present in some instrument's own list.


In [6]:
filter_lists = inst["filters"].map(json.loads)
lengths = filter_lists.map(len)
print(f"instruments with >=1 filter: {(lengths > 0).sum()} of {len(inst)}")
all_labels = sorted({label for one_list in filter_lists for label in one_list})
print(f"distinct raw filter labels: {len(all_labels)}")
lower_groups = pd.Series(all_labels).groupby(pd.Series(all_labels).str.casefold()).agg(list)
variant_groups = {k: v for k, v in lower_groups.items() if len(v) > 1}
print(f"case/whitespace variant groups among filter labels: {variant_groups or 'none'}")

grandma_photometry_filters = GRANDMA.loc[GRANDMA["section"] == "Photometry", "filter"]
print(f"\nGRANDMA rows with a Photometry-shaped Filter value: "
      f"{has_content(grandma_photometry_filters).sum()} of {len(GRANDMA)}")

obs_filters = set(obs["filt"].dropna())
print(f"\ndistinct historical observation filt labels: {sorted(obs_filters)}")
print(f"present in some instrument's own filters list: {obs_filters.issubset(set(all_labels))}")


instruments with >=1 filter: 91 of 95
distinct raw filter labels: 56
case/whitespace variant groups among filter labels: none

GRANDMA rows with a Photometry-shaped Filter value: 32 of 39

distinct historical observation filt labels: ['bessellr', 'ps1::open']
present in some instrument's own filters list: True


## Decision 6 — morning / evening

**Observed.** Recomputed across all 89 telescopes: 81 carry a
quoted-timestamp-shaped `morning`/`evening` value and 8 carry the literal
JSON boolean `false`, exactly matching which telescopes lack a fixed,
known location. Stage 4 traced this to SkyPortal `v1.4.0` source
(`Telescope.current_time`): both fields are the *next* astronomical
(-18 degree) twilight computed live via `astroplan` at request time, and
the API returns literal `False` for both when the site has no fixed
location — not a stored schedule.
**Why it matters.** A value computed relative to "now" and frozen into
one capture instant is stale the moment the capture finishes; presenting
it as a stored property of the telescope would let a downstream reader
mistake a one-off computed timestamp for current or future twilight, or
for a live availability signal.
**Decided.** `morning`/`evening` are excluded from the static
observational-resource capability record in this first version. They are
not transformed or normalized here because they are not retained at all;
if a later stage needs twilight information it must recompute it on
demand from the telescope's site coordinates (a `DERIVED` value with an
explicit timestamp), never read these two frozen fields as current state.
**Scope.** 89 of 89 telescopes affected (81 timestamp-shaped, 8 literal
`false`); 0 telescopes carry the literal `true`.


In [7]:
def morning_category(value):
    if value == "false":
        return "json-bool false"
    if value == "true":
        return "json-bool true"
    if isinstance(value, str) and value.startswith('"'):
        return "quoted timestamp string"
    return f"other: {value!r}"

categories = tel["morning"].map(morning_category)
print("morning value categories, recomputed across all 89 telescopes:")
print(categories.value_counts().to_string())

has_fixed_location = tel["fixed_location"] & has_content(tel["lat"])
predicted_false = ~has_fixed_location
actual_false = tel["morning"] == "false"
print(f"\n'not fixed_location or no lat/lon' predicts the literal false value on "
      f"{int((predicted_false == actual_false).sum())} of {len(tel)} telescopes")
print(f"evening matches morning's category on every row: "
      f"{(tel['evening'].map(morning_category) == categories).all()}")


morning value categories, recomputed across all 89 telescopes:
morning
quoted timestamp string    81
json-bool false             8

'not fixed_location or no lat/lon' predicts the literal false value on 89 of 89 telescopes
evening matches morning's category on every row: True


## Decision 7 — FOV / region

**Observed.** ICARE's own footprint evidence (`region`/`region_summary`)
is populated on 19 of 95 instruments. GRANDMA's `fov_deg` is populated on
32 of 39 rows (all Photometry-section; Spectroscopy-section rows report
`'-'`). Of the 33 GRANDMA rows Decision 2 judged enrichment-eligible, 26
map to an ICARE telescope with exactly one instrument (safe to associate
a telescope-level FOV with that one instrument) and 7 map to a telescope
with more than one instrument (e.g. Canada-France-Hawaii Telescope hosts
2 ICARE instruments matched by 2 different GRANDMA rows with different
FOVs) — a telescope-level value cannot be assigned to a specific
instrument there without more evidence than a name match provides.
**Why it matters.** ICARE's own region text is instrument-specific and
directly measured; GRANDMA's FOV is telescope-level and, on a
multi-instrument telescope, ambiguous about which camera it describes.
Assigning it to every instrument on that telescope would silently invent
instrument-level precision the evidence does not support.
**Decided.** ICARE `region`/`region_summary` remains the preferred,
`DIRECT`, instrument-level footprint evidence wherever populated.
GRANDMA `fov_deg` may supplement, as `COMPLEMENTARY_REFERENCE`, only for
EXACT/HIGH-CONFIDENCE CANDIDATE matches (Decision 2) whose ICARE
telescope hosts exactly one instrument; for multi-instrument telescopes
the GRANDMA FOV is retained at the telescope level only and is not
automatically attached to any specific instrument. AMBIGUOUS/UNMATCHED
GRANDMA rows contribute no FOV enrichment. An instrument with neither
ICARE region content nor an eligible single-instrument GRANDMA match has
`UNKNOWN` footprint — never interpolated from another instrument.
**Scope.** 19 of 95 ICARE instruments carry direct region evidence; 26 of
33 enrichment-eligible GRANDMA rows are safe for instrument-level FOV
association, 7 must stay telescope-level only; 7 of 39 GRANDMA rows
(AMBIGUOUS + UNMATCHED) contribute no FOV evidence at all.


In [8]:
region_present = has_content(inst["region"])
print(f"ICARE instruments with populated region/region_summary: {int(region_present.sum())} of {len(inst)}")

grandma_fov_present = has_content(GRANDMA["fov_deg"])
print(f"GRANDMA rows with a populated fov_deg: {int(grandma_fov_present.sum())} of {len(GRANDMA)}")
fov_present_by_section = has_content(GRANDMA["fov_deg"]).groupby(GRANDMA["section"]).sum()
print(fov_present_by_section.to_string())

print(f"\nsingle-instrument enrichment-eligible matches (instrument-level FOV OK): "
      f"{single_instrument} of {len(GRANDMA_ELIGIBLE)}")
print(f"multi-instrument enrichment-eligible matches (telescope-level FOV only): "
      f"{multi_instrument} of {len(GRANDMA_ELIGIBLE)}")

no_evidence = int((~region_present).sum()) - single_instrument
print(f"\ninstruments with neither ICARE region content nor an eligible single-instrument GRANDMA "
      f"match remain UNKNOWN for footprint (upper bound, before checking overlap): "
      f"up to {int((~region_present).sum())} of {len(inst)} lack ICARE region content")


ICARE instruments with populated region/region_summary: 19 of 95
GRANDMA rows with a populated fov_deg: 32 of 39
section
Photometry      32
Spectroscopy     0

single-instrument enrichment-eligible matches (instrument-level FOV OK): 26 of 33
multi-instrument enrichment-eligible matches (telescope-level FOV only): 7 of 33

instruments with neither ICARE region content nor an eligible single-instrument GRANDMA match remain UNKNOWN for footprint (upper bound, before checking overlap): up to 76 of 95 lack ICARE region content


## Decision 8 — sensitivity / Mlim

**Observed.** Four genuinely different kinds of "how deep can this see"
evidence exist. (1) ICARE `sensitivity_data` (static, `DIRECT`): populated
on 2 of 95 instruments, a single filter key (`'ps1::open'`). (2) GRANDMA
`mlim` (`COMPLEMENTARY_REFERENCE`): populated on all 39 rows, documented
in the caption as "the typical maximum limiting mag obtained in less than
1 h" — a nominal figure under a stated exposure ceiling, not a guarantee;
eligible for enrichment on 33 rows (Decision 2). (3) Historical
observation `limmag` (`EMPIRICAL`): populated on all 93 observations, but
those 93 observations cover only 3 of 95 instruments
(`instrument_id` 9, 22, 23), using only 2 filters; `exposure_time` takes
only 2 distinct values (120 s, 180 s); `airmass` and `seeing` are
populated on 0 of 93 rows, contributing nothing. (4) A "current
guaranteed detectability" figure: no such field or computation exists
anywhere in this evidence.
**Why it matters.** Conflating any of the first three with the fourth
would let the resource layer or the LLM claim a specific instrument can
currently reach a certain depth, when the truth is either a nominal
network-wide figure, a different instrument's historical result, or
nothing at all.
**Decided.** Keep the three available kinds of evidence distinct and
explicitly labeled as above; do not compute or expose a "current
guaranteed detectability" value in this stage. Missing `sensitivity_data`
is never inferred from another instrument, its telescope, or a network
average. `EMPIRICAL` limmag is scoped strictly to the instrument that was
actually observed (see Decision 11). `COMPLEMENTARY_REFERENCE` Mlim
follows Decision 2's matching-confidence and Decision 7's
telescope/instrument-level rules. Where none of the three exist for an
instrument, sensitivity is `UNKNOWN` — an expected, valid state, not an
error.
**Scope.** ICARE sensitivity_data: 2 of 95 instruments. GRANDMA Mlim: 39
of 39 rows populated, 33 enrichment-eligible. Historical limmag: 93 of 93
observations, but only 3 of 95 instruments represented; airmass/seeing:
0 of 93.


In [9]:
sensitivity_columns = [c for c in inst.columns if c.startswith("sensitivity_data.")]
has_sensitivity = inst[sensitivity_columns].apply(has_content).any(axis=1)
print(f"ICARE instruments with any sensitivity_data content: {int(has_sensitivity.sum())} of {len(inst)}")
print(inst.loc[has_sensitivity, ["id", "name"] + sensitivity_columns].to_string(index=False))

print(f"\nGRANDMA mlim coverage: {int(has_content(GRANDMA['mlim']).sum())} of {len(GRANDMA)}")
print(f"GRANDMA mlim, enrichment-eligible rows only: {len(GRANDMA_ELIGIBLE)}")

print(f"\nhistorical observations: {len(obs)} rows")
print(f"distinct instrument_id represented: {sorted(obs['instrument_id'].unique())} "
      f"({obs['instrument_id'].nunique()} of {len(inst)} instruments)")
print(f"limmag coverage: {int(has_content(obs['limmag']).sum())} of {len(obs)}")
print(f"distinct filt: {sorted(obs['filt'].unique())}")
print(f"distinct exposure_time: {sorted(obs['exposure_time'].unique())}")
print(f"airmass coverage: {int(has_content(obs['airmass']).sum())} of {len(obs)}")
print(f"seeing coverage: {int(has_content(obs['seeing']).sum())} of {len(obs)}")

per_instrument = obs.groupby("instrument_id")["limmag"].agg(["count", "min", "median", "max"])
print("\nempirical limmag, per represented instrument only:")
print(per_instrument.to_string())


ICARE instruments with any sensitivity_data content: 2 of 95
 id      name sensitivity_data.ps1::open.magsys  sensitivity_data.ps1::open.zeropoint  sensitivity_data.ps1::open.exposure_time sensitivity_data.ps1::open.limiting_magnitude
  7 TAROT/TCA                                ab                                  28.2                                     180.0                                          19.0
  9 TAROT/TRE                                ab                                  26.3                                      30.0                                            18

GRANDMA mlim coverage: 39 of 39
GRANDMA mlim, enrichment-eligible rows only: 33

historical observations: 93 rows
distinct instrument_id represented: [np.int64(9), np.int64(22), np.int64(23)] (3 of 95 instruments)
limmag coverage: 93 of 93
distinct filt: ['bessellr', 'ps1::open']
distinct exposure_time: [np.int64(120), np.int64(180)]
airmass coverage: 0 of 93
seeing coverage: 0 of 93

empirical limmag, per repres

## Decision 9 — allocation / access

**Observed.** 38 allocations reference 34 of 95 instruments. The raw
allocation record's own keys are
`{pi, proposal_id, group_id, instrument_id, hours_allocated,
default_share_group_ids, types, validity_ranges, allocation_users,
created_at, modified, instrument, id}` — none named availability, active,
status, or online. Telescope id=137 nests allocation id=78 in its
serialized `instruments[].allocations`, but no row `id=78` exists in the
standalone allocations table (re-confirmed from the raw JSON in Decision
1); Stage 4 traced a plausible but not certain explanation
(group-scoped permission filtering: allocation 78 belongs to
`group_id=122`).
**Why it matters.** `hours_allocated` looks numeric and "operational"
enough to be mistaken for a live capacity signal; without an explicit
decision, a deterministic filter or the LLM could wrongly treat "has an
allocation" as "is available right now".
**Decided.** An allocation establishes `AUTHORIZATION`/`ACCESS` only — a
Group/PI holds an hours budget on one Instrument — and must never be
read as evidence of current, real-time availability; no field on the
record represents live state. The telescope-137/allocation-78 nested-vs-
standalone discrepancy's root cause remains `NOT ESTABLISHED`; it is
preserved as an open, flagged discrepancy, not repaired, deleted, or
silently reconciled in either direction.
**Scope.** 38 allocations, 34 of 95 instruments hold >=1 allocation; 0 of
38 standalone records carry an availability-like field; 1 of 38 nested
allocation references has no standalone counterpart.


In [10]:
print(f"allocations: {len(alloc)} rows, referencing {alloc['instrument_id'].nunique()} of "
      f"{len(inst)} instruments")

with (ICARE_RAW_DIR / "allocations.json").open() as handle:
    raw_allocation_records = json.load(handle)["data"]
all_keys = sorted({key for record in raw_allocation_records for key in record})
print(f"\nraw allocation record keys: {all_keys}")
availability_like = [k for k in all_keys if re.search(r"avail|active|status|online", k, re.I)]
print(f"availability-like keys found: {availability_like or 'none'}")

print(f"\ntelescope id=137 nested allocation id=78 present in standalone table: "
      f"{78 in set(alloc['id'])}")
with (ICARE_RAW_DIR / "telescopes.json").open() as handle:
    raw_telescope_records = json.load(handle)["data"]
nested_78 = next(a for t in raw_telescope_records if t["id"] == 137
                 for i in t["instruments"] for a in i.get("allocations", []) if a["id"] == 78)
print(f"allocation id=78's group_id (from the nested copy, since no standalone row exists): "
      f"{nested_78['group_id']}")
print("root cause: NOT ESTABLISHED (plausible group-scoped permission filtering, per Stage 4; "
      "not traced to certainty)")


allocations: 38 rows, referencing 34 of 95 instruments

raw allocation record keys: ['allocation_users', 'created_at', 'default_share_group_ids', 'group_id', 'hours_allocated', 'id', 'instrument', 'instrument_id', 'modified', 'pi', 'proposal_id', 'types', 'validity_ranges']
availability-like keys found: none

telescope id=137 nested allocation id=78 present in standalone table: False
allocation id=78's group_id (from the nested copy, since no standalone row exists): 122
root cause: NOT ESTABLISHED (plausible group-scoped permission filtering, per Stage 4; not traced to certainty)


## Decision 10 — Use and Rob

**Observed.** GRANDMA's caption documents `Use` as a usage-frequency
category for GRANDMA operations (`'VF'`/`'F'`/`'R'`/`'VR'`/`'nOP'`) and
`Rob` as "if the telescope is robotic or not". Recomputed: 36 of 39 `Use`
values fall inside that documented vocabulary (3 do not — see Decision
13). `Rob` takes 6 distinct raw values (`no`, `yes`, `remote`, `np`,
`semi`, `nOP`) — richer than the documented yes/no concept. Comparing
`Rob` against ICARE's `robotic` boolean on the 33 enrichment-eligible
telescopes: most agree, but `TRT-SBO` disagrees outright (ICARE
`robotic=False`, GRANDMA `rob='yes'`), and several eligible rows carry a
`Rob` value (`remote`/`np`) a boolean cannot represent.
**Why it matters.** `Use` names a frequency-of-observation category for
GRANDMA's own operations, not a live usage or availability signal; a
disagreement on `Rob`, and values it cannot even express as a boolean,
mean ICARE's `robotic` cannot simply be replaced by GRANDMA's `Rob`
without losing or misrepresenting information.
**Decided.** `Use` is retained only as `COMPLEMENTARY_REFERENCE`
metadata, per its documented meaning; it is never surfaced as a current-
availability or current-usage claim. `Rob` may complement ICARE's
`robotic` (as `COMPLEMENTARY_REFERENCE`) only for enrichment-eligible
telescopes (Decision 2) where the two plainly `AGREE`; where they
`DIFFER` (e.g. `TRT-SBO`) or GRANDMA's value is not a plain yes/no, both
raw values are preserved side by side with a conflict/richer-value flag,
per Decision 3's general policy — ICARE's own `robotic` remains the
value shown by default, GRANDMA's alongside it, never silently
overwritten.
**Scope.** `Use`: 39 of 39 GRANDMA rows populated, 36 within the
documented vocabulary. `Rob`: 39 of 39 populated; of 33 enrichment-
eligible telescopes, recomputed counts of AGREES / DIFFERS / NOT
DIRECTLY COMPARABLE below.


In [11]:
print("Use: raw value counts vs the documented vocabulary (VF/F/R/VR/nOP):")
print(GRANDMA["use"].value_counts().to_string())
outside_vocab = ~GRANDMA["use"].isin({"VF", "F", "R", "VR", "nOP"})
print(f"outside the documented vocabulary: {int(outside_vocab.sum())} of {len(GRANDMA)}")

print("\nRob: raw value counts:")
print(GRANDMA["rob"].value_counts().to_string())

rob_rows = []
for _, m in GRANDMA_ELIGIBLE.iterrows():
    grandma_row = GRANDMA[GRANDMA["row_index_in_source"] == m["row_index_in_source"]].iloc[0]
    icare_row = tel[tel["name"] == m["icare_name"]].iloc[0]
    g_rob = grandma_row["rob"]
    icare_robotic = bool(icare_row["robotic"])
    if g_rob not in ("yes", "no"):
        status = "NOT DIRECTLY COMPARABLE"
    else:
        status = "AGREES" if (g_rob == "yes") == icare_robotic else "DIFFERS"
    rob_rows.append({"grandma_name": m["grandma_name"], "icare_name": m["icare_name"],
                     "icare_robotic": icare_robotic, "grandma_rob": g_rob, "status": status})
rob_comparison = pd.DataFrame(rob_rows)
print(f"\nRob vs robotic, {len(rob_comparison)} enrichment-eligible telescopes:")
print(rob_comparison["status"].value_counts().to_string())
print("\nDIFFERS rows:")
print(rob_comparison[rob_comparison["status"] == "DIFFERS"].to_string(index=False))


Use: raw value counts vs the documented vocabulary (VF/F/R/VR/nOP):
use
VR      13
F        8
R        7
VF       5
nOP      3
BV R     1
O        1
nOp      1
outside the documented vocabulary: 3 of 39

Rob: raw value counts:
rob
no        28
yes        7
remote     1
np         1
semi       1
nOP        1

Rob vs robotic, 33 enrichment-eligible telescopes:
status
AGREES                     26
NOT DIRECTLY COMPARABLE     4
DIFFERS                     3

DIFFERS rows:
grandma_name                   icare_name  icare_robotic grandma_rob  status
     TRT-SBO Thai Robotic Telescope - SBO          False         yes DIFFERS
        T120                     OHP/T120           True          no DIFFERS
     TRT-SRO Thai Robotic Telescope - SRO          False         yes DIFFERS


## Decision 11 — historical observations

**Observed.** 93 historical observations exist, covering only 3 of 95
instruments (`instrument_id` 9, 22, 23); `limmag` reaches 100% coverage
on those 93 rows, `exposure_time` takes 2 distinct values, `filt` takes 2
distinct values, and `airmass`/`seeing` reach 0% coverage.
**Why it matters.** Any per-instrument or per-telescope statistic drawn
from these observations describes only 3 instruments; presenting it as
representative of the other 92 (or of their telescopes generally) would
fabricate coverage the data do not have.
**Decided.** Historical observations are retained as `EMPIRICAL`
supporting evidence, strictly scoped to the 3 instruments actually
observed. They are excluded from the first static capability record for
every other instrument — no generalization across instruments or
telescopes. A later stage may summarize them (e.g. per-instrument
min/median/max `limmag`), but any such summary must stay attached to that
specific instrument's `id` and must not be presented as a network-wide or
telescope-class capability.
**Scope.** 93 observations across 3 of 95 instruments; 2 distinct `filt`
labels; `limmag` populated on 93 of 93; `airmass` and `seeing` populated
on 0 of 93.


In [12]:
print(f"historical observations: {len(obs)}")
print(f"distinct instruments represented: {obs['instrument_id'].nunique()} of {len(inst)} "
      f"({sorted(obs['instrument_id'].unique())})")
print(f"distinct filt labels: {sorted(obs['filt'].unique())}")
print(f"limmag coverage: {int(has_content(obs['limmag']).sum())} of {len(obs)}")
print(f"airmass coverage: {int(has_content(obs['airmass']).sum())} of {len(obs)}")
print(f"seeing coverage: {int(has_content(obs['seeing']).sum())} of {len(obs)}")

coverage_fraction = obs["instrument_id"].nunique() / len(inst)
print(f"\nfraction of the ICARE instrument population these observations could ever speak to: "
      f"{coverage_fraction:.1%} ({obs['instrument_id'].nunique()} of {len(inst)})")


historical observations: 93
distinct instruments represented: 3 of 95 ([np.int64(9), np.int64(22), np.int64(23)])
distinct filt labels: ['bessellr', 'ps1::open']
limmag coverage: 93 of 93
airmass coverage: 0 of 93
seeing coverage: 0 of 93

fraction of the ICARE instrument population these observations could ever speak to: 3.2% (3 of 95)


## Decision 12 — important negative / scope exclusions

**Observed.** Across all 4 ICARE tables' columns, none represents current
weather or current queue state; `telescopes.weather_link` is a URL string,
populated on 3 of 89 telescopes, never resolved or parsed in this
evidence. Decisions 6, 8, 9 and 10 above each independently found that no
field in this evidence represents live, current, guaranteed state.
**Why it matters.** Without an explicit statement, a consumer could
assume the resource layer implicitly covers real-time operational status
because Stage 1 scoped that out only in passing.
**Decided.** The observational-resource layer built from this evidence
does not claim current weather, current queue state, guaranteed real-time
availability, or guaranteed detectability at any magnitude, for any
telescope or instrument. `weather_link` is retained only as an opaque URL
reference, never resolved into weather data by this project.
**Scope.** 0 of the 4 ICARE tables' columns represent live weather, queue
state, or availability; `weather_link` populated on 3 of 89 telescopes.


In [13]:
all_columns = {name: list(frame.columns) for name, frame in ICARE.items()}
live_state_like = {
    name: [c for c in columns if re.search(r"weather(?!_link)|queue|available|is_online|live_status", c, re.I)]
    for name, columns in all_columns.items()
}
print("columns matching a live weather/queue/availability pattern (excluding weather_link itself):")
for name, columns in live_state_like.items():
    print(f"  {name:14s} {columns or 'none'}")

print(f"\nweather_link coverage: {int(has_content(tel['weather_link']).sum())} of {len(tel)}")
print("sample weather_link values (URLs, not weather data):")
print(tel.loc[has_content(tel["weather_link"]), ["name", "weather_link"]].to_string(index=False))


columns matching a live weather/queue/availability pattern (excluding weather_link itself):
  telescopes     none
  instruments    none
  allocations    none
  observations   none

weather_link coverage: 3 of 89
sample weather_link values (URLs, not weather data):
      name                                               weather_link
   Colibri         http://www.cleardarksky.com/c/PdrMrtrObMBCkey.html
UBAI/NT-60           https://app.weathercloud.net/d5359208089#current
   GoChile http://clearoutside.com/forecast/-30.47/-70.76?view=midday


## Decision 13 — uncertain GRANDMA cells

**Observed.** Re-applying the documented `Use` vocabulary check to the
current GRANDMA interim table finds the same 3 flagged cells the
extractor reported: row 2 (`TNT` at Xinglong Obs.) carries `use='BV R'`,
which looks like a column-boundary reconstruction ambiguity; row 27
(`Perkin-Elmer Tel.`) carries `use='O'`; row 34 (`Terskol- 2m/MMCS`)
carries `use='nOp'`. All three are preserved exactly as extracted; none
was silently repaired.
**Why it matters.** A decision made from an unresolved cell, treated as
if it were a clean value, would be no more trustworthy than a decision
made up outright.
**Decided.** None of the 3 cells is repaired here or elsewhere in this
notebook. Row 2's `use` value is excluded from any `Use`-based decision
or enrichment (treated as `UNKNOWN` for `Use` only); its other columns
are unaffected. Rows 27 and 34 are genuine raw source values/case
variants, not extraction errors, and are preserved verbatim as
`COMPLEMENTARY_REFERENCE` evidence without being folded into the 5-value
documented vocabulary.
**Scope.** 3 of 39 GRANDMA rows affected, in their `use` cell only; 0
other columns affected for these 3 rows. 1 of the 3 (row 2, `TNT` at
Xinglong Obs., matched to `Xinglong-TNT`) is otherwise among the 33
enrichment-eligible telescopes used elsewhere in this notebook — only its
`Use` evidence is withheld, its `Rob`/`Size`/`FOV`/`Mlim` evidence is
unaffected; the other 2 (rows 27, 34) are UNMATCHED/AMBIGUOUS and
contribute no enrichment regardless.


In [14]:
DOCUMENTED_USE_VOCABULARY = {"VF", "F", "R", "VR", "nOP"}
flagged = GRANDMA.loc[~GRANDMA["use"].isin(DOCUMENTED_USE_VOCABULARY),
                      ["row_index_in_source", "telescope_name", "section", "use", "filter"]]
print(f"cells outside the documented Use vocabulary: {len(flagged)} of {len(GRANDMA)}")
print(flagged.to_string(index=False))

affected_rows = set(flagged["row_index_in_source"])
overlap_with_eligible = GRANDMA_ELIGIBLE[GRANDMA_ELIGIBLE["row_index_in_source"].isin(affected_rows)]
print(f"\nof these 3 flagged rows, {len(overlap_with_eligible)} are among the "
      f"{len(GRANDMA_ELIGIBLE)} enrichment-eligible telescopes used elsewhere in this notebook:")
print(overlap_with_eligible[["row_index_in_source", "grandma_name", "icare_name"]].to_string(index=False))


cells outside the documented Use vocabulary: 3 of 39
 row_index_in_source    telescope_name      section  use      filter
                   2               TNT   Photometry BV R   g1r1i1 BV
                  27 Perkin-Elmer Tel.   Photometry    O UBV R I C C
                  34  Terskol- 2m/MMCS Spectroscopy  nOp 3800 ´ 9000

of these 3 flagged rows, 1 are among the 33 enrichment-eligible telescopes used elsewhere in this notebook:
 row_index_in_source grandma_name   icare_name
                   2          TNT Xinglong-TNT


## What this record establishes

Each of the thirteen decisions above answers to evidence measured fresh
in this notebook, independently of `A_eda.ipynb` and
`A2_grandma_evidence.ipynb`. Three themes run through them. Some
decisions establish a durable identity and provenance framework
(Decisions 1-3) that every later field-specific decision then applies.
Several decide what a piece of evidence is *not* allowed to be read as —
`morning`/`evening` are not current state, an allocation is not
availability, `Use` is not a live signal, historical observations do not
generalize beyond the 3 instruments that produced them. And several
decide, explicitly, that the honest answer for many instruments is
`UNKNOWN`: sensitivity, footprint, and GRANDMA enrichment all leave most
of the ICARE population without a value rather than inventing one.

No decision here resolves a conflict by picking a winner without
evidence; where ICARE and GRANDMA disagree (aperture, robotic status),
both raw values survive with their origin labeled. No decision merges
telescope-level GRANDMA information onto every instrument of a
multi-instrument telescope. The three uncertain GRANDMA cells and the
telescope-137/allocation-78 discrepancy are carried forward as open,
flagged uncertainty rather than being resolved by assumption.

### Final decision summary

| # | Area | Decision | Main evidence | Implementation consequence |
|---|------|----------|----------------|------------------------------|
| 1 | Resource identity | ICARE numeric `id` is the sole canonical identity for telescopes/instruments/allocations/observations; nested embedded relations are supporting, not canonical; names are never used for internal identity. | 0 duplicate ids across 4 tables; 0 dangling FKs in 3 relations; 1 nested-vs-standalone discrepancy; GRANDMA reuses `'TNT'` for 2 telescopes. | Build the resource layer keyed on ICARE `id`; never join or dedupe ICARE records by `name`. |
| 2 | GRANDMA matching | Automatic enrichment only from `EXACT`/`HIGH-CONFIDENCE CANDIDATE` GRANDMA rows, matched per row not per name. | 10 EXACT + 23 HIGH-CONFIDENCE / 3 AMBIGUOUS / 3 UNMATCHED, of 39 GRANDMA rows. | Implementation must carry `GRANDMA_MATCHES`-equivalent logic and reject AMBIGUOUS/UNMATCHED rows from auto-enrichment. |
| 3 | Provenance & conflicts | Adopt `DIRECT`/`COMPLEMENTARY_REFERENCE`/`EMPIRICAL`/`DERIVED`/`UNKNOWN`; on disagreement, keep both values, flag conflict, canonical stays `UNKNOWN` unless a specific decision states otherwise. | Real DIFFERS found in size and Rob among the 33 eligible telescopes. | Every enriched/derived field needs an origin column and, where applicable, a conflict flag; no silent overwrite. |
| 4 | Band | Retain raw `band`; add a casefolded value for grouping only. | 90 of 95 instruments affected by the `Optical`/`optical` case split. | Do not replace `band`; add one derived comparison column. |
| 5 | Filters | ICARE instrument-level `filters` is primary; GRANDMA `filter` stays reference-only, no cross-vocabulary mapping. | 91 of 95 instruments have filters (56 labels); GRANDMA uses an unrelated convention. | Filter queries must use `instruments.filters`; GRANDMA filter text is descriptive metadata only. |
| 6 | Morning/evening | Excluded from the static capability record; must be recomputed on demand if ever needed, never read from this capture. | 89 of 89 telescopes carry a live-computed value (81 timestamp / 8 `false`). | Do not carry `morning`/`evening` into the resource layer. |
| 7 | FOV/region | ICARE region preferred; GRANDMA FOV supplements only single-instrument, enrichment-eligible telescopes; multi-instrument telescopes stay telescope-level only. | 19 of 95 instruments have ICARE region; 26 of 33 eligible matches are single-instrument. | Enrichment logic must check instrument count per matched telescope before attaching FOV. |
| 8 | Sensitivity/Mlim | Keep ICARE static, GRANDMA nominal, and empirical historical depth distinct; never infer across instruments; no "current guaranteed" value computed. | 2 of 95 instruments (ICARE); 39 of 39 GRANDMA rows; 93 obs across 3 of 95 instruments. | Resource schema needs 3 separate sensitivity-evidence fields, each labeled, none used to fill another. |
| 9 | Allocation/access | Allocation = authorization/access only, never availability; id-78 discrepancy left unresolved. | 38 allocations / 34 of 95 instruments; 0 availability-like fields; 1 unresolved nested reference. | Deterministic filter/LLM must not treat "has an allocation" as "is available now". |
| 10 | Use/Rob | `Use` is reference-only, not a usage/availability signal; `Rob` complements `robotic` only where they agree. | 36 of 39 Use values in vocabulary; Rob DIFFERS at least once (`TRT-SBO`) among 33 eligible. | Store `robotic` (DIRECT) and `rob` (COMPLEMENTARY_REFERENCE) side by side with a conflict flag. |
| 11 | Historical observations | Retained as empirical evidence scoped to the 3 instruments observed; excluded from the general capability record. | 93 observations, 3 of 95 instruments, limmag 100%, airmass/seeing 0%. | Any future summary must stay keyed to those 3 instrument ids only. |
| 12 | Negative scope | No claim of current weather, queue state, real-time availability, or guaranteed detectability. | 0 of 4 tables' columns represent live state; `weather_link` populated on 3 of 89, URL only. | Resource layer documentation/schema must state these exclusions explicitly. |
| 13 | Uncertain GRANDMA cells | Preserve raw/flagged; exclude the affected value from enrichment rather than repair it. | 3 of 39 GRANDMA rows flagged in `use` only; 1 otherwise enrichment-eligible. | Enrichment code must check the flagged-cell list before using `use`. |
